In [ ]:
USE SCHEMA CLINICAL_LAKEHOUSE.STAGING;

In [ ]:
CREATE OR REPLACE STORAGE INTEGRATION s3_healthcare_int
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'S3'
  ENABLED = TRUE
  STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::695843249921:role/SnowflakeStorageRole'
  STORAGE_ALLOWED_LOCATIONS = ('s3://vc-healthcare-datalake/');

In [ ]:
DESCRIBE INTEGRATION s3_healthcare_int;

In [ ]:
CREATE OR REPLACE STAGE s3_fhir_stage
  STORAGE_INTEGRATION = s3_healthcare_int
  URL = 's3://vc-healthcare-datalake/standardized/fhir/'
  FILE_FORMAT = (TYPE = JSON);

In [ ]:
LIST @s3_fhir_stage;

In [ ]:
COPY INTO staging.stg_fhir_patients (src_json)
FROM @s3_fhir_stage/patients/
FILE_FORMAT = (TYPE = JSON);

In [ ]:
COPY INTO staging.stg_fhir_medications (src_json)
FROM @s3_fhir_stage/medications/
FILE_FORMAT = (TYPE = JSON);

In [ ]:
COPY INTO staging.stg_fhir_encounters (src_json)
FROM @s3_fhir_stage/encounters/
FILE_FORMAT = (TYPE = JSON);

In [ ]:
COPY INTO staging.stg_fhir_conditions (src_json)
FROM @s3_fhir_stage/conditions/
FILE_FORMAT = (TYPE = JSON);

In [ ]:
SELECT
    src_json:id::STRING AS patient_id,
    src_json:gender::STRING AS gender
FROM staging.stg_fhir_patients
LIMIT 5;